In [1]:
!pip -q install fair-esm pandas numpy requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 3.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import requests
from io import StringIO

url = (
    "https://raw.githubusercontent.com/"
    "szczurek-lab/ampdiffusion-starter-kit/"
    "main/experimental/mic.csv"
)

r = requests.get(url)
r.raise_for_status()

mic = pd.read_csv(StringIO(r.text))

print("Rows:", len(mic))
print("Peptides:", mic["peptide_id"].nunique())
print("Columns:", list(mic.columns))

# Potent = MIC <= 16 µM on at least one strain
potent = (
    mic[
        (mic["mic_relation"] == "=") &
        (mic["mic"] <= 16)
    ]
    [["peptide_id", "sequence"]]
    .drop_duplicates("sequence")
    .reset_index(drop=True)
)

print("\nPotent reference peptides:", len(potent))
print(potent.head())

Rows: 506
Peptides: 46
Columns: ['peptide_id', 'sequence', 'modification', 'strain', 'strain_type', 'mic', 'mic_unit', 'mic_relation', 'medium', 'cfu_per_ml', 'ph', 'salt']

Potent reference peptides: 33
    peptide_id                    sequence
0  AMP_diff2-3             MWRLKKWLRLLRWRF
1  AMP_diff2-4  MKFKKILKIGLGKLKNLALKLLKVLK
2  AMP_diff2-5                MFRKIWRIHRIL
3  AMP_diff2-8            MFARPIARKILRALFK
4  AMP_diff2-9          LWRLLRRIGRLILRGFRW


In [3]:
import torch
import esm
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
model = model.eval().to(device)

batch_converter = alphabet.get_batch_converter()

REP_LAYER = 6
EMBED_DIM = 320

def embed_sequences(sequences, batch_size=256):

    all_vecs = []

    for start in range(0, len(sequences), batch_size):

        batch_seqs = sequences[start:start+batch_size]

        data = [
            (f"seq_{i}", seq)
            for i, seq in enumerate(batch_seqs)
        ]

        _, strs, tokens = batch_converter(data)
        tokens = tokens.to(device)

        with torch.no_grad():
            out = model(
                tokens,
                repr_layers=[REP_LAYER],
                return_contacts=False
            )

        reps = out["representations"][REP_LAYER]

        vecs = []

        for i, seq in enumerate(strs):
            L = len(seq)

            vec = (
                reps[i, 1:L+1]
                .mean(dim=0)
                .cpu()
                .numpy()
            )

            vecs.append(vec)

        all_vecs.append(np.asarray(vecs))

    return np.vstack(all_vecs)


potent_embeddings = embed_sequences(
    potent["sequence"].tolist()
)

print("Potent embedding matrix:", potent_embeddings.shape)

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt
Potent embedding matrix: (33, 320)


In [4]:
from google.colab import files

uploaded = files.upload()
candidate_file = next(iter(uploaded))

cand = pd.read_csv(candidate_file, low_memory=False)

print("Rows:", len(cand))
print("Unique:", cand["sequence"].nunique())

assert len(cand) == 60000
assert cand["sequence"].nunique() == 60000

Saving STEP9_60K_EMBEDDING_METADATA.csv to STEP9_60K_EMBEDDING_METADATA.csv
Rows: 60000
Unique: 60000


In [7]:
import time
import numpy as np
import torch

def embed_sequences(sequences, batch_size=256):

    all_vecs = []
    times = []

    total = len(sequences)

    for start in range(0, total, batch_size):

        end = min(start + batch_size, total)
        batch_seqs = sequences[start:end]

        data = [
            (f"seq_{i}", seq)
            for i, seq in enumerate(batch_seqs)
        ]

        _, strs, tokens = batch_converter(data)
        tokens = tokens.to(device)

        t0 = time.time()

        with torch.no_grad():
            out = model(
                tokens,
                repr_layers=[REP_LAYER],
                return_contacts=False
            )

        reps = out["representations"][REP_LAYER]

        vecs = []

        for i, seq in enumerate(strs):
            L = len(seq)

            vec = (
                reps[i, 1:L+1]
                .mean(dim=0)
                .cpu()
                .numpy()
            )

            vecs.append(vec)

        all_vecs.append(
            np.asarray(vecs, dtype=np.float32)
        )

        elapsed = time.time() - t0
        times.append(elapsed)

        avg = np.mean(times)

        remaining_batches = (
            total - end + batch_size - 1
        ) // batch_size

        eta = avg * remaining_batches

        print(
            f"{end:,}/{total:,} | "
            f"batch {elapsed:.2f}s | "
            f"ETA {eta/60:.1f} min",
            flush=True
        )

    return np.vstack(all_vecs)

In [8]:
candidate_embeddings = embed_sequences(
    cand["sequence"].tolist(),
    batch_size=256
)

print(
    "Candidate embedding matrix:",
    candidate_embeddings.shape
)

256/60,000 | batch 2.73s | ETA 10.7 min
512/60,000 | batch 3.06s | ETA 11.3 min
768/60,000 | batch 3.24s | ETA 11.6 min
1,024/60,000 | batch 2.63s | ETA 11.2 min
1,280/60,000 | batch 2.65s | ETA 11.0 min
1,536/60,000 | batch 2.70s | ETA 10.8 min
1,792/60,000 | batch 3.82s | ETA 11.3 min
2,048/60,000 | batch 2.72s | ETA 11.1 min
2,304/60,000 | batch 2.67s | ETA 11.0 min
2,560/60,000 | batch 2.65s | ETA 10.8 min
2,816/60,000 | batch 3.31s | ETA 10.9 min
3,072/60,000 | batch 3.09s | ETA 10.9 min
3,328/60,000 | batch 3.02s | ETA 10.9 min
3,584/60,000 | batch 2.60s | ETA 10.8 min
3,840/60,000 | batch 3.10s | ETA 10.8 min
4,096/60,000 | batch 3.20s | ETA 10.8 min
4,352/60,000 | batch 2.70s | ETA 10.7 min
4,608/60,000 | batch 2.66s | ETA 10.6 min
4,864/60,000 | batch 2.70s | ETA 10.5 min
5,120/60,000 | batch 3.64s | ETA 10.6 min
5,376/60,000 | batch 2.68s | ETA 10.5 min
5,632/60,000 | batch 2.68s | ETA 10.4 min
5,888/60,000 | batch 2.64s | ETA 10.3 min
6,144/60,000 | batch 3.47s | ETA 10.3 mi

In [11]:
from sklearn.preprocessing import normalize

# Normalize embeddings
cand_norm = normalize(
    candidate_embeddings.astype(np.float32)
)

potent_norm = normalize(
    potent_embeddings.astype(np.float32)
)

# Candidate x reference cosine similarity
similarity = cand_norm @ potent_norm.T

# Maximum similarity to any known potent peptide
cand["esm2_potent_max_cosine"] = similarity.max(axis=1)

# Mean Top-5 similarity
k = min(5, similarity.shape[1])

top5 = np.partition(
    similarity,
    -k,
    axis=1
)[:, -k:]

cand["esm2_potent_mean_top5_cosine"] = (
    top5.mean(axis=1)
)

# Potent centroid
centroid = potent_embeddings.mean(axis=0)
centroid = centroid / np.linalg.norm(centroid)

cand["esm2_potent_centroid_cosine"] = (
    cand_norm @ centroid
)

# Rank: higher biological similarity = better
cand["esm2_potent_rank"] = (
    cand["esm2_potent_mean_top5_cosine"]
    .rank(
        ascending=False,
        method="first"
    )
    .astype(int)
)

cand[
    [
        "sequence",
        "esm2_potent_max_cosine",
        "esm2_potent_mean_top5_cosine",
        "esm2_potent_centroid_cosine",
        "esm2_potent_rank"
    ]
].head()

,sequence,esm2_potent_max_cosine,esm2_potent_mean_top5_cosine,esm2_potent_centroid_cosine,esm2_potent_rank
0,IWEELLRKKLLAAAKWKMMAWWWW,0.951040,0.939694,0.931550,21518
1,FLLEELLLKWAMAAAKWWKLRWWWW,0.949315,0.934198,0.928426,24526
2,GAFGAIKLIWKSLKWWWWWWW,0.901527,0.881583,0.879996,45215
3,LHYRIILMRLAKWWWWWWWWW,0.922930,0.906719,0.881836,37095
4,FIHIKIIKNNALLLKKWWWWWWWWW,0.907346,0.901836,0.908176,38948


In [12]:
OUT = "/content/STEP9B_60K_WITH_BIOLOGICAL_EMBEDDING_SCORES.csv"

cand.to_csv(
    OUT,
    index=False
)

print("Saved:", OUT)

print("\nSummary:")
print(
    cand[
        [
            "esm2_potent_max_cosine",
            "esm2_potent_mean_top5_cosine",
            "esm2_potent_centroid_cosine"
        ]
    ].describe()
)

Saved: /content/STEP9B_60K_WITH_BIOLOGICAL_EMBEDDING_SCORES.csv

Summary:
       esm2_potent_max_cosine  esm2_potent_mean_top5_cosine  \
count            60000.000000                  60000.000000   
mean                 0.924786                      0.907532   
std                  0.051035                      0.057060   
min                  0.634739                      0.609857   
25%                  0.904122                      0.882445   
50%                  0.938876                      0.923500   
75%                  0.961696                      0.949634   
max                  0.990043                      0.983232   

       esm2_potent_centroid_cosine  
count                 60000.000000  
mean                      0.878164  
std                       0.067686  
min                       0.516665  
25%                       0.852631  
50%                       0.899635  
75%                       0.923930  
max                       0.981870  


In [13]:
from google.colab import files
files.download(
    "/content/STEP9B_60K_WITH_BIOLOGICAL_EMBEDDING_SCORES.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>